# Price Variance Recovery Pipeline — DuckDB engine (v2, feedback-revised)

Rerun addressing Daria's feedback (8/13). **No logic changes to the recovery math** — fixes are
output format, field sets/ordering, and the blank-MPN rule.

**Changes in v2**
1. **Blank/NULL MPN skipped** before matching → no price, no opportunity, absent from all outputs
   (fixes the 191k-vs-10k evidence issue and the spurious 4.5 contracted price).
2. **Delimiter corruption fixed** — outputs written as **.xlsx** (quoted **.csv** too). Embedded commas
   in Line Description / Supplier no longer shift columns.
3. **enriched_ap** — all 20 fields in Daria's exact order, incl. carried-through
   `contractno`, `contractdescription`, selected price start/end dates.
4. **recovery_case_evidence** — requested source fields added, sequence matched.
5. **product_review_list** — adds supplier, manufacturer (mfrName), biggest-spend line description
   (line with max Extended Amount), total spend, total quantity.

**Config:** MPN = Supplier Item Identifier only · hierarchy LOCAL → ALLIED → CLINERGY →
PREMIER-SURPASS → PREMIER-SPECIALS → GPO → PREMIER-NATIONAL → 832 · blank contractType ignored.

## 0. Install (run once)

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [80 lines of output]
  running bdist_wheel
  running build
  running build_py
  creating build
  creating build\lib.win-amd64-cpython-37
  creating build\lib.win-amd64-cpython-37\duckdb
  copying duckdb\bytes_io_wrapper.py -> build\lib.win-amd64-cpython-37\duckdb
  copying duckdb\filesystem.py -> build\lib.win-amd64-cpython-37\duckdb
  copying duckdb\udf.py -> build\lib.win-amd64-cpython-37\duckdb
  copying duckdb\__init__.py -> build\lib.win-amd64-cpython-37\duckdb
  creating build\lib.win-amd64-cpython-37\duckdb\typing
  copying duckdb\typing\__init__.py -> build\lib.win-amd64-cpython-37\duckdb\typing
  creating build\lib.win-amd64-cpython-37\duckdb\query_graph
  copying duckdb\query_graph\__main__.py -> build\lib.win-amd64-cpython-37\duckdb\query_graph
  creating build\lib.win-amd64-cpython-37\duckdb\functional
  copying duckdb\functional\__init__.py -> build\lib.win-amd64-cpython-37\duckdb\functional
  creating build\lib.


  Using cached duckdb-1.3.2.tar.gz (11.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build duckdb


## 1. Imports & config

c:\program files\python37\python.exe
3.7.0 (v3.7.0:1bf9cc5093, Jun 27 2018, 04:59:51) [MSC v.1914 64 bit (AMD64)]


In [1]:
import os
import numpy as np
import pandas as pd
import duckdb
from sqlalchemy import create_engine, text, bindparam

CONTRACT_HIERARCHY = [
    "LOCAL", "ALLIED", "CLINERGY", "PREMIER-SURPASS",
    "PREMIER-SPECIALS", "GPO", "PREMIER-NATIONAL", "832",
]
ANALYSIS_START = "2025-01-01"
ANALYSIS_END   = "2026-06-30"
LOW_PRICE_RATIO = 0.5

## 2. MySQL connection  ⬅️ **edit here**

In [2]:
DB_USER     = "admin"
DB_PASSWORD = os.getenv("DB_PASSWORD", "Gpoproddb!#!")
DB_HOST     = "prod-db.c969yoyq9cyy.us-east-1.rds.amazonaws.com"
DB_PORT     = 3306
DB_NAME     = "inmar"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)
with engine.connect() as c:
    print("Connected:", c.exec_driver_sql("SELECT DATABASE()").scalar())

Connected: inmar


## 3. Load AP (filtered) — now carrying **all** source fields Daria's spec needs

Added vs v1: `Supplier ID`, `Supplier's Invoice Number`, `Purchase Orders`, `Item`, `Unit Cost`.
Money `text` columns cast to DECIMAL in SQL. **Blank/NULL MPN dropped here** so it never matches.

In [6]:
ap_sql = """
SELECT
    `Supplier Item Identifier` AS MPN,
    `Supplier`,
    `Supplier ID`,
    `Invoice Number`,
    `Supplier's Invoice Number`,
    `Invoice Date`                  AS invoice_date,
    CAST(NULLIF(REPLACE(REPLACE(`Invoice Amount`,',',''),'$',''),'') AS DECIMAL(18,4)) AS invoice_amount,
    `Purchase Orders`,
    `Line Description`,
    `Item`,
    `Unit of Measure`,
    CAST(NULLIF(REPLACE(REPLACE(`Extended Amount`,',',''),'$',''),'') AS DECIMAL(18,4)) AS extended_amount,
    `Quantity` AS quantity,
    `Invoice Status`
FROM Invoice_detail_1
WHERE LOWER(TRIM(`Invoice Status`)) NOT IN ('cancelled','denied')
  AND `Invoice Date`    BETWEEN %(start)s AND %(end)s
  AND `Supplier Item Identifier` IS NOT NULL
  AND TRIM(`Supplier Item Identifier`) <> ''
"""
ap = pd.read_sql(ap_sql, engine, params={"start": ANALYSIS_START, "end": ANALYSIS_END})
ap["MPN"] = ap["MPN"].astype(str).str.strip()
print("AP rows (MPN present, in-window):", len(ap))
ap.head(3)

AP rows (MPN present, in-window): 312879


,MPN,Supplier,Supplier ID,Invoice Number,Supplier's Invoice Number,invoice_date,invoice_amount,Purchase Orders,Line Description,Item,Unit of Measure,extended_amount,quantity,Invoice Status
0,M00513332,7138 Boston Scientific Corporation,7138,SINV-00340993,704521040,2025-01-30,363.55,20177655,FORCEPS BIOPSY OVAL CUP SERRATED WITH NEEDLE L...,10073850 - FRCP BX OVAL RADIAL JAW 4 240,Box,363.55,1.0,Approved
1,PLI3803Z,1580 Medline Industries Inc,1580,SINV-00340991,2355345986,2025-01-30,82.85,20176695,COVER PROBE PRE GELLED ON INSIDE FASTER SETUP ...,NaN,Box,82.85,1.0,Approved
2,SSN100255Z,1580 Medline Industries Inc,1580,SINV-00340986,2355357242,2025-01-30,55.82,20168509,"Needles: Sure-Snap Hypodermic Safety Needle, 2...",NaN,Box,55.82,2.0,Approved


## 4. Load Novus — pre-filtered by AP MPN set; carry `contractno`, `contractdescription`, `mfrName`

In [12]:
mpns = ap["MPN"].dropna().unique().tolist()
print("distinct MPNs:", len(mpns))

def load_novus_for(mpn_list, batch=1000):
    base = """
    SELECT catNo, uom, contractType, price,
           contractno, contractdescription, mfrName,
             createdAt AS created_at,
           priceEffDate AS price_eff,
          priceExpDate AS price_exp
    FROM HunterAICCHS_ContractPriceRepository_post2025_20260625_2
    WHERE contractType IS NOT NULL AND TRIM(contractType) <> ''
      AND catNo IN :mpns

    """
    q = text(base).bindparams(bindparam("mpns", expanding=True))
    frames = []
    for i in range(0, len(mpn_list), batch):
        frames.append(pd.read_sql(q, engine, params={
            "mpns": mpn_list[i:i+batch], "start": ANALYSIS_START, "end": ANALYSIS_END}))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

novus = load_novus_for(mpns)
print("Novus rows (filtered):", len(novus))
novus.head(3)

distinct MPNs: 53793
Novus rows (filtered): 499466


,catNo,uom,contractType,price,contractno,contractdescription,mfrName,created_at,price_eff,price_exp
0,R01695,PK,GPO,6.1002,PP-LA-579,Manual Microbiology,Remel Inc.,2025-01-02,2024-05-14,2025-08-31
1,BR1223650,PK,GPO,168.8300,AD-OR-2081,Instrument Cleaners Enzymatics,STERIS Corporation,2025-01-02,2023-02-01,2025-01-16
2,BR1223650,PK,GPO,168.8300,AD-OR-2081,Instrument Cleaners Enzymatics,STERIS Corporation,2025-01-02,2023-02-01,2025-01-16


## 5. Register in DuckDB

In [13]:
con = duckdb.connect()          # file-backed: duckdb.connect("pv.duckdb") to spill to disk
con.execute("PRAGMA threads=4")
con.register("ap", ap)
con.register("novus", novus)
con.register("hierarchy",
    pd.DataFrame({"contractType": CONTRACT_HIERARCHY,
                  "priority": range(1, len(CONTRACT_HIERARCHY)+1)}))

## 6. Clean AP + build effective periods

MPN already non-blank (filtered in SQL). Unit Price Paid = round(abs(extended/qty),2), NULL when qty 0/blank.

In [14]:
con.execute("""
CREATE OR REPLACE TEMP TABLE ap_clean AS
SELECT *,
       CASE WHEN quantity IS NULL OR quantity = 0 THEN NULL
            ELSE round(abs(extended_amount / quantity), 2) END AS unit_price_paid
FROM ap;

CREATE OR REPLACE TEMP TABLE novus_periods AS
SELECT catNo, uom, contractType, price, contractno, contractdescription, mfrName,
       greatest(created_at, price_eff) AS start_dt,
       least(
         coalesce(lead(created_at) OVER (
           PARTITION BY catNo, uom, contractType ORDER BY created_at), price_exp),
         price_exp) AS end_dt
FROM novus;
""")
print("periods:", con.execute("SELECT count(*) FROM novus_periods").fetchone()[0])

periods: 499466


## 7. Range join → pick by hierarchy → opportunity, **carrying contract metadata**

Selected price now brings through `contractno`, `contractdescription`, `start_dt`, `end_dt`
so enriched fields 15–20 can populate.

In [15]:
con.execute("""
CREATE OR REPLACE TEMP TABLE enriched AS
WITH matched AS (
    SELECT a.*, p.contractType, p.price AS contracted_price,
           p.contractno, p.contractdescription, p.mfrName,
           p.start_dt AS price_start_date, p.end_dt AS price_end_date,
           h.priority
    FROM ap_clean a
    JOIN novus_periods p
      ON p.catNo = a.MPN
     AND a.invoice_date >= p.start_dt
     AND a.invoice_date <  p.end_dt
    JOIN hierarchy h ON h.contractType = p.contractType
),
ranked AS (
    SELECT *, row_number() OVER (
        PARTITION BY "Invoice Number", MPN, invoice_date, extended_amount
        ORDER BY priority) AS rn
    FROM matched
),
best AS (SELECT * FROM ranked WHERE rn = 1),
final AS (
    SELECT a.*,
           b.contractType        AS contracted_price_source,
           b.contracted_price, b.contractno, b.contractdescription,
           b.mfrName, b.price_start_date, b.price_end_date
    FROM ap_clean a
    LEFT JOIN best b
      ON  b."Invoice Number" = a."Invoice Number"
      AND b.MPN = a.MPN
      AND b.invoice_date = a.invoice_date
      AND b.extended_amount IS NOT DISTINCT FROM a.extended_amount
)
SELECT *,
       CASE
         WHEN quantity IS NULL OR quantity = 0 THEN 0
         WHEN contracted_price IS NULL THEN 0
         WHEN unit_price_paid > contracted_price
           THEN abs(quantity) * (unit_price_paid - contracted_price) * sign(coalesce(invoice_amount,0))
         ELSE 0
       END AS opportunity
FROM final;
""")
n = con.execute("SELECT count(*) FROM enriched").fetchone()[0]
tot = con.execute("SELECT round(sum(opportunity),2) FROM enriched").fetchone()[0]
print("enriched rows:", n, "| total opportunity:", tot)

enriched rows: 312879 | total opportunity: 16207545.46


## 8. enriched_ap — Daria's exact 20-field order

Fields 1–16 as required; 17–20 (contract no/desc, price start/end) carried through.

In [16]:
enriched_ap = con.execute("""
SELECT
    MPN                        AS "Supplier Item Identifier",
    Supplier                   AS "Supplier",
    "Invoice Number"           AS "Invoice Number",
    "Supplier's Invoice Number" AS "Supplier's Invoice Number",
    invoice_date               AS "Invoice Date",
    invoice_amount             AS "Invoice Amount",
    "Purchase Orders"          AS "Purchase Orders",
    "Line Description"         AS "Line Description",
    "Item"                     AS "Item",
    "Unit of Measure"          AS "Unit of Measure",
    extended_amount            AS "Extended Amount",
    quantity                   AS "Quantity",
    MPN                        AS "MPN",
    unit_price_paid            AS "Unit Price Paid",
    contracted_price           AS "Novus Contracted Price",
    opportunity                AS "Price Variance Opportunity",
    contractno                 AS "Novus Contract Number",
    contractdescription        AS "Novus Contract Description",
    price_start_date           AS "Price Start Date",
    price_end_date             AS "Price End Date"
FROM enriched
ORDER BY "Invoice Number", "Supplier Item Identifier"
""").df()
enriched_ap.head(5)

,Supplier Item Identifier,Supplier,Invoice Number,Supplier's Invoice Number,Invoice Date,Invoice Amount,Purchase Orders,Line Description,Item,Unit of Measure,Extended Amount,Quantity,MPN,Unit Price Paid,Novus Contracted Price,Price Variance Opportunity,Novus Contract Number,Novus Contract Description,Price Start Date,Price End Date
0,081139021,CH011291 Performance Health Supply Inc,SINV-00326280,IN98353295,2025-01-01,55.65,20167692,BANDAGE COMPRESSION COTTON LATEX FREE 6CMX5MTR...,10139255 - BANDAGE COMPRESSION COTTON LATEX FR...,Roll,47.70,10.0,081139021,4.77,NaN,0.0,NaN,NaN,NaN,NaN
1,081139153,CH011291 Performance Health Supply Inc,SINV-00326290,IN98353288,2025-01-01,62.07,20167695,MOLLELAST CON BANDAGE 6CMX4M,NaN,Package,54.12,6.0,081139153,9.02,NaN,0.0,NaN,NaN,NaN,NaN
2,PRE4010,D00943 KCI USA,SINV-00326662,33085855,2025-01-02,1357.29,20164572,Prevena™ Plus 125 Therapy Unit (14-Day)-PRE401...,NaN,Each,1357.29,3.0,PRE4010,452.43,NaN,0.0,NaN,NaN,NaN,NaN
3,H7493893103J2,7138 Boston Scientific Corporation,SINV-00326850,703992581,2025-01-02,4209.86,20168123,GUIDEWIRE PT2 185CM 014 MODERATE SUPPORT J,10023700 - GWR PT2 185CM 014 MDRT SUPP J,Box,1050.00,3.0,H7493893103J2,350.00,NaN,0.0,NaN,NaN,NaN,NaN
4,H7493918912250,7138 Boston Scientific Corporation,SINV-00326850,703992581,2025-01-02,4209.86,20168123,CATHETER ANGIOPLASTY RAPID EXCHANGE EMERGE MON...,10068115 - CATHETER ANGIOPLASTY RAPID EXCHANGE...,Each,80.00,1.0,H7493918912250,80.00,NaN,0.0,NaN,NaN,NaN,NaN


## 9. recovery_case_evidence — positive-opportunity lines, requested field sequence

All rows have an MPN by construction (blank MPN was filtered upstream).

In [17]:
evidence = con.execute("""
SELECT
    MPN                        AS "Supplier Item Identifier",
    Supplier                   AS "Supplier",
    "Invoice Number"           AS "Invoice Number",
    "Supplier's Invoice Number" AS "Supplier's Invoice Number",
    invoice_date               AS "Invoice Date",
    invoice_amount             AS "Invoice Amount",
    "Purchase Orders"          AS "Purchase Orders",
    "Line Description"         AS "Line Description",
    "Item"                     AS "Item",
    "Unit of Measure"          AS "Unit of Measure",
    extended_amount            AS "Extended Amount",
    quantity                   AS "Quantity",
    MPN                        AS "MPN",
    unit_price_paid            AS "Unit Price Paid",
    contracted_price           AS "Novus Contracted Price",
    opportunity                AS "Price Variance Opportunity",
    contractno                 AS "Novus Contract Number",
    contractdescription        AS "Novus Contract Description",
    price_start_date           AS "Price Start Date",
    price_end_date             AS "Price End Date"
FROM enriched
WHERE opportunity > 0
ORDER BY opportunity DESC
""").df()
print("evidence rows:", len(evidence))
evidence.head(5)

evidence rows: 10588


,Supplier Item Identifier,Supplier,Invoice Number,Supplier's Invoice Number,Invoice Date,Invoice Amount,Purchase Orders,Line Description,Item,Unit of Measure,Extended Amount,Quantity,MPN,Unit Price Paid,Novus Contracted Price,Price Variance Opportunity,Novus Contract Number,Novus Contract Description,Price Start Date,Price End Date
0,866066_NAM,D07444 Philips Healthcare,SINV-00412022,9028291877,2025-06-05,680659.49,20152218,INTELLIVUE MX550 US AL1 MX550 ADVANCED MONITOR,NaN,Each,805484.46,53.0,866066_NAM,15197.82,0.00,805484.46,PP-NS-1948,Physiological Monitoring Sys,2025-01-06,2026-01-28
1,353535,SUP-00007502 Mako Surgical Corp,SINV-00533109,108920,2025-12-30,713000.00,20313323,Stryker Robotic Arm System 4 (MakoTM/RIO),NaN,Kit,713000.00,1.0,353535,713000.00,645.00,712355.00,AHG-OR-0004,Surgical Instruments,2025-04-09,2026-01-31
2,2,D02482 Intuitive Surgical,SINV-00373873,905869223,2025-02-27,575000.00,20192591,Da Vinci 5 Console,NaN,Each,575000.00,1.0,2,575000.00,20.44,574979.56,AD-NS-1283,Peak Use Rental Equipment,2025-01-02,2025-11-30
3,867036,D07444 Philips Healthcare,SINV-00520402,9060113071,2025-12-15,1225195.53,20264854,IntelliVue MMX,NaN,Each,404059.58,67.0,867036,6030.74,0.00,404059.58,PP-NS-1948,Physiological Monitoring Sys,2025-01-06,2026-01-28
4,867036,D07444 Philips Healthcare,SINV-00412019,9028291876,2025-06-05,419332.59,20152218,INTELLIVUE MMX,NaN,Each,340250.46,53.0,867036,6419.82,0.00,340250.46,PP-NS-1948,Physiological Monitoring Sys,2025-01-06,2026-01-28


## 10. product_review_list — added source rollups

Per MPN: supplier + manufacturer + biggest-spend line description (line with max Extended Amount)
+ total spend + total quantity, alongside the opportunity aggregates.

In [18]:
product_review = con.execute("""
WITH base AS (
    SELECT MPN,
           sum(opportunity)                                 AS total_opportunity,
           count(*)                                         AS transaction_count,
           sum(CASE WHEN opportunity > 0 THEN 1 ELSE 0 END) AS opportunity_line_count,
           avg(unit_price_paid)                             AS avg_unit_price_paid,
           avg(contracted_price)                            AS avg_contracted_price,
           sum(extended_amount)                             AS total_spend,
           sum(quantity)                                    AS total_quantity,
           any_value(Supplier)                              AS supplier,
           any_value(mfrName)                               AS manufacturer
    FROM enriched
    GROUP BY MPN
),
-- description of the line with the highest Extended Amount per MPN
top_desc AS (
    SELECT MPN, "Line Description" AS biggest_spend_description
    FROM (
        SELECT MPN, "Line Description",
               row_number() OVER (PARTITION BY MPN ORDER BY extended_amount DESC) rn
        FROM enriched
    ) WHERE rn = 1
)
SELECT b.MPN,
       b.supplier,
       b.manufacturer,
       t.biggest_spend_description,
       b.total_spend,
       b.total_quantity,
       b.transaction_count,
       b.opportunity_line_count,
       b.avg_unit_price_paid,
       b.avg_contracted_price,
       b.total_opportunity,
       CASE WHEN b.avg_unit_price_paid > 0
             AND b.avg_contracted_price < ? * b.avg_unit_price_paid
            THEN 'Contracted price much lower than paid - possible product match / UOM issue.'
            ELSE '' END AS comments
FROM base b LEFT JOIN top_desc t USING (MPN)
ORDER BY b.total_opportunity DESC
""", [LOW_PRICE_RATIO]).df()
product_review.head(10)

,MPN,supplier,manufacturer,biggest_spend_description,total_spend,total_quantity,transaction_count,opportunity_line_count,avg_unit_price_paid,avg_contracted_price,total_opportunity,comments
0,866066_NAM,D07444 Philips Healthcare,Philips Healthcare,INTELLIVUE MX550 US AL1 MX550 ADVANCED MONITOR,1561957.16,104.0,5,5.0,14776.650000,0.000000,1561957.16,Contracted price much lower than paid - possib...
1,867036,D07444 Philips Healthcare,Philips Healthcare,IntelliVue MMX,1490887.30,251.0,11,11.0,5850.187273,0.000000,1490887.30,Contracted price much lower than paid - possib...
2,470179,D02482 Intuitive Surgical,Encore Medical Device Repair LLC,SCISSORS ROBOTIC MONOPOLAR CURVED JAW OPENING ...,1387680.00,413.0,110,102.0,3360.000000,224.000000,1201088.00,Contracted price much lower than paid - possib...
3,353535,SUP-00007502 Mako Surgical Corp,Teleflex LLC,Stryker Robotic Arm System 4 (MakoTM/RIO),723531.00,2.0,2,1.0,361765.500000,668.000000,712355.00,Contracted price much lower than paid - possib...
4,471205,D02482 Intuitive Surgical,Encore Medical Device Repair LLC,FORCEPS ROBOTIC BIPOLAR FENESTRATED HANDPIECE ...,690900.00,235.0,70,70.0,2940.000000,167.714286,650240.00,Contracted price much lower than paid - possib...
5,2,D03963 Village Green Tropic Interiors,"US Med-Equip, LLC",Da Vinci 5 Console,588320.00,90.0,22,20.0,26717.142273,22.529091,587608.78,Contracted price much lower than paid - possib...
6,563392,D02657 Guldmann Inc,"Performance Health Supply, LLC","GH3+ 250 1I (550 lbs) Wi-FI, Class III Scale, ...",535208.90,122.0,3,3.0,4392.956667,181.910000,513015.88,Contracted price much lower than paid - possib...
7,866064_NAM,D07444 Philips Healthcare,Philips Healthcare,INTELLIVUE MX500 US AL1 MX500 ADVANCED MONITOR,336405.96,27.0,2,2.0,12459.480000,0.000000,336405.96,Contracted price much lower than paid - possib...
8,2037556-001,7838 Datex - Ohmeda,"GE Medical Systems Information Technologies, Inc.",259cx Fetal Monitor Model,301686.60,20.0,2,2.0,15084.330000,0.000000,301686.60,Contracted price much lower than paid - possib...
9,3603,7511 Skytron,"LSL Industries, LLC dba LSL Healthcare",ULTRA SLIDE,297000.00,6.0,4,4.0,49500.000000,161.940000,296028.36,Contracted price much lower than paid - possib...


## 11. Export — .xlsx **and** .csv (quoted), no delimiter corruption

In [19]:
import csv
def export(df, name):
    df.to_excel(f"{name}.xlsx", index=False)
    df.to_csv(f"{name}.csv", index=False, quoting=csv.QUOTE_ALL, encoding="utf-8-sig")
    print(f"wrote {name}.xlsx + {name}.csv  ({len(df):,} rows)")

export(enriched_ap,    "enriched_ap")
export(evidence,       "recovery_case_evidence")
export(product_review, "product_review_list")

wrote enriched_ap.xlsx + enriched_ap.csv  (312,879 rows)
wrote recovery_case_evidence.xlsx + recovery_case_evidence.csv  (10,588 rows)
wrote product_review_list.xlsx + product_review_list.csv  (53,793 rows)
